# 05 — Controlled candidate training and MLflow evidence

**Objectives**

- Fit two declared changes on the exact same training rows.
- Compare them on the exact same validation rows.
- Log explicit inputs, model IDs, signatures, input examples, and the lock.

**Prerequisite:** lesson 04.


In [ ]:
import pandas as pd

from aai_local_classification.learning import short_digest
from aai_local_classification.workflow import run_candidate_selection
from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings

settings = load_settings()
root = study_root()
print(f"Course state: {root}")
print(f"Experiment: {settings.experiment_name}")


In [ ]:
selection = run_candidate_selection(settings, root)
comparison = pd.DataFrame(
    [
        {
            "candidate": item.candidate_name,
            "run_id": item.run_id,
            "model_id": item.model_id,
            "validation_average_precision": item.threshold_selection.validation_metrics.average_precision,
            "validation_roc_auc": item.threshold_selection.validation_metrics.roc_auc,
            "validation_brier": item.threshold_selection.validation_metrics.brier_score,
        }
        for item in selection.candidates
    ]
).sort_values("validation_average_precision", ascending=False)
comparison


In [ ]:
print(f"Selected candidate: {selection.selected_candidate}")
print(f"Selected model ID: {selection.selected_model_id}")
print(f"Dataset digest: {short_digest(selection.dataset_sha256)}")
print(f"Selection rule: {selection.selection_rule}")
assert selection.primary_metric == "average_precision"


The change is controlled: only the estimator family differs. Both candidates
receive the same declared features and train/validation partitions. A
predeclared tolerance prefers the simpler model when average precision is
practically tied, avoiding needless complexity for a tiny validation advantage.
The selected model is a first-class MLflow Logged Model (`models:/<model-id>`),
not a path we guess from the run layout.

The course logs explicitly rather than registering inside `fit()`: exploratory
training should not create registry versions. The model uses a representative
input example, inferred signature, and `skops` serialization; load artifacts
only from trusted sources even with a safer format.

### Exercise

Add a hyperparameter change to one candidate. Which facts must remain fixed for
the comparison to support a causal explanation of the score change?

**Hint:** data/split, feature contract, preprocessing, seed policy, metric
definition, and threshold procedure are experimental controls.

**Checkpoint:** candidate selection used validation average precision only; the
test file was never loaded by the training workflow.

Next: **06_model_selection_and_threshold.ipynb**.
